# Beam Collimation 

When moving from fiber to free space, parameters such as the mode field diameter and numerical aperature can be found from the spec sheets.The numerical aperture (NA) is a dimensionless number that characterizes the range of angles over which the system can accept or emit light. It is definted as: $$NA = nsin\theta_1$$ as shown in the figure below. We see here that $\theta_1$ is the half-angle of the maximum cone of light that can enter or exit the lens. 

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/4c/Numerical_aperture.svg/500px-Numerical_aperture.svg.png" width="300">


This code allows us to find the right focal length of an aspheric lens to collimate a beam given a desired beam diameter and known beam divergence of the light being emitted from the fiber to free space. Beam divergence can be calculated if one knows the numerical aperature, or the mode field diameter (MFD) of the fiber. Solving the previous equation for the half-angle: $$\theta_1 = arcsin(NA)$$ This is half of the full beam divergence, $\phi$.

From here we simply multiply whatever $\theta_1$ we get by two and have the beam divergence to insert into the code. The code calculates the following equation for focal length: $$f = \frac{\frac{\phi}{2}}{tan(\frac{\theta}{2})}$$

More information:
- On collimation: https://www.thorlabs.com/insights/?tabName=Collimation 
- On fiber collimation/coupling: https://www.thorlabs.com/insights/?tabName=Fiber%20Optics 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import c, h, hbar

# Collimate diverging beam

In [2]:
# thorlabs collimation how-to example: https://www.thorlabs.com/newgrouppage9.cfm?objectgroup_id=1832

def lens_for_collimation(beam_div_full_deg, target_beam_diameter_m):
    '''Calculate the focal length of a lens needed to collimate a beam with a given divergence beam_div_deg to a target_beam_diameter_m.
    beam_div_deg: Full angle beam divergence in degrees (e.g., 6 deg)'''

    # choose fast axis or else for round beam, common beam divergence, photodigm~6deg
    beam_div_full_rad = beam_div_full_deg / 180 * np.pi 

    # here beam_divergence is the full-angle beam divergence, so that beam_div/2 is related to radius over focal length
    target_beam_radius_m = target_beam_diameter_m / 2
    beam_div_half_rad = beam_div_full_rad / 2

    # geometry: tan(beam_divergence_half) = beam_waist_radius / focal_len 
    focal_len = target_beam_radius_m / (np.tan(beam_div_half_rad))

    print(f"Focal length to collimate beam full-divergence of {beam_div_full_deg:.2f} deg to target beam diameter of {target_beam_diameter_m*1e3:.2f} mm:")
    print(f"    {focal_len*1e3:.3f} mm")
    print()
    return focal_len

print("testing based on thorlabs example...")
lens_for_collimation(30, 3e-3);

testing based on thorlabs example...
Focal length to collimate beam full-divergence of 30.00 deg to target beam diameter of 3.00 mm:
    5.598 mm



In [3]:
# f=4.55mm lens (mounted asphere C230TMD-B), would give beam spot diameter between 475-650um 
lens_for_collimation(6, 0.475e-3);  
lens_for_collimation(8, 0.64e-3);

# f=2.00mm lens (mounted asphere C151TMD-B), would give beam spot diameter 200-300um  
lens_for_collimation(6, .21e-3); 
lens_for_collimation(8, .28e-3);

Focal length to collimate beam full-divergence of 6.00 deg to target beam diameter of 0.47 mm:
    4.532 mm

Focal length to collimate beam full-divergence of 8.00 deg to target beam diameter of 0.64 mm:
    4.576 mm

Focal length to collimate beam full-divergence of 6.00 deg to target beam diameter of 0.21 mm:
    2.004 mm

Focal length to collimate beam full-divergence of 8.00 deg to target beam diameter of 0.28 mm:
    2.002 mm



# Fiber collimation - pick collimating asphere

In [4]:
# from https://www.thorlabs.com/newgrouppage9.cfm?objectgroup_id=14204 , far field approximation, uses fiber MFD to calculate beam divergence

def fiber_beam_divergence(MFD, wavelen=532e-9):
    """Calculate the full-beam divergence of a Gaussian beam exiting a fiber.
    MFD: Mode Field Diameter in meters
    wavelen: Wavelength in meters
    Returns: Beam divergence in radians (half angle)
    """
    beam_div_full_rad = wavelen / (np.pi * MFD)     # half the full angular extent of the beam, wavelen-dependent 

    print(f"Fiber with MFD = {MFD*1e6:.2f} um and {wavelen*1e9 = :.0f}nm has a full-beam divergence of {beam_div_full_rad/np.pi*180:.2f} deg")
    
    return beam_div_full_rad

fiber_beam_divergence(MFD=6.4e-6, wavelen=1064e-9) /np.pi*180
# fiber_beam_divergence(MFD=3.3e-6, wavelen=532e-9) /np.pi*180


def collimate_fiber_to_beam_diameter(MFD, wavelen, target_beam_diameter_m):
    """Calculate the focal length of a lens needed to collimate a beam exiting a fiber with a given MFD and wavelength to a target beam diameter.
    MFD: Mode Field Diameter in meters
    wavelen: Wavelength in meters
    target_beam_diameter_m: Desired collimated beam diameter in meters
    Returns: Focal length in millimeters
    """
    beam_div_full_rad = fiber_beam_divergence(MFD, wavelen)
    asphere_focal_len = lens_for_collimation(beam_div_full_rad/np.pi*180, target_beam_diameter_m)
    
    return asphere_focal_len*1e3 # return in mm

Fiber with MFD = 6.40 um and wavelen*1e9 = 1064nm has a full-beam divergence of 3.03 deg


## Test for collimating 1064 nm regular SM fibers to various target beam diameters
Stock 1064 nm fiber from thorlabs:
- https://www.thorlabs.com/single-mode-fcapc-fiber-optic-patch-cables?tabName=Overview
- https://media.thorlabs.com/globalassets/items/s/sm/sm9/sm980-5.8-125/ttn027884-s01.pdf?v=0116030002 

In [5]:
# calculate what asphere to collimate fiber beam to target beam diameter 

collimate_fiber_to_beam_diameter(MFD=4e-6, wavelen=1064e-9, target_beam_diameter_m=200e-6)
collimate_fiber_to_beam_diameter(MFD=5e-6, wavelen=1064e-9, target_beam_diameter_m=200e-6)
collimate_fiber_to_beam_diameter(MFD=6.4e-6, wavelen=1064e-9, target_beam_diameter_m=200e-6)

Fiber with MFD = 4.00 um and wavelen*1e9 = 1064nm has a full-beam divergence of 4.85 deg
Focal length to collimate beam full-divergence of 4.85 deg to target beam diameter of 0.20 mm:
    2.361 mm

Fiber with MFD = 5.00 um and wavelen*1e9 = 1064nm has a full-beam divergence of 3.88 deg
Focal length to collimate beam full-divergence of 3.88 deg to target beam diameter of 0.20 mm:
    2.951 mm

Fiber with MFD = 6.40 um and wavelen*1e9 = 1064nm has a full-beam divergence of 3.03 deg
Focal length to collimate beam full-divergence of 3.03 deg to target beam diameter of 0.20 mm:
    3.778 mm



3.7784775584547567